# Phase 4 — Complete SAR workflow integration

This executable submission notebook demonstrates the same state machine used by the FastAPI application:

`CSV → CaseData → Risk Analyst → Human risk gate → Compliance Officer → Human compliance gate → SAR JSON`

The workflow below exercises the state machine. Compliance generation is called only after risk approval, and filing occurs only after final approval.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "data").is_dir() and (cwd / "src").is_dir():
    STARTER = cwd
elif cwd.name == "notebooks":
    STARTER = cwd.parent
else:
    STARTER = cwd / "starter"
sys.path.insert(0, str(STARTER))

from src.compliance_officer_agent import ComplianceOfficerAgent
from src.foundation_sar import DataLoader, ExplainabilityLogger
from src.risk_analyst_agent import RiskAnalystAgent
from src.sar_workflow import HumanDecisionInput, SarWorkflow

OUTPUTS = STARTER / "outputs"
AUDIT = OUTPUTS / "audit_logs" / "phase4_notebook.jsonl"
AUDIT.parent.mkdir(parents=True, exist_ok=True)
if AUDIT.exists():
    AUDIT.unlink()

logger = ExplainabilityLogger(AUDIT)
loader = DataLoader.load(STARTER / "data")
workflow = SarWorkflow(
    loader,
    RiskAnalystAgent(None, logger),
    ComplianceOfficerAgent(None, logger),
    logger,
    OUTPUTS,
)
print(loader.summary().model_dump())

{'customer_count': 150, 'account_count': 178, 'transaction_count': 4268}


## Approved path — Stage 1 risk analysis

The risk agent analyzes `CUST_0053` and stops at the first human gate. At this point no compliance narrative exists.

In [2]:
risk_record = workflow.analyze_customer("CUST_0053")
print({
    "stage": risk_record.stage,
    "classification": risk_record.risk_analysis.suspicious_activity_type,
    "confidence": risk_record.risk_analysis.confidence_score,
    "compliance_output": risk_record.compliance_output,
})
assert risk_record.stage == "awaiting_risk_review"
assert risk_record.compliance_output is None

{'stage': 'awaiting_risk_review', 'classification': 'Structuring', 'confidence': 0.93, 'compliance_output': None}


## Human gate 1 — Approve compliance generation

A named reviewer records a rationale. Only this approval permits the Compliance Officer Agent to run.

In [3]:
risk_decision = HumanDecisionInput(
    decision="Approved",
    reviewer="Notebook Reviewer",
    rationale="Repeated cash deposits in the under-$10,000 band require a factual narrative review",
)
compliance_record = workflow.review_risk(risk_record.case.case_id, risk_decision)
print({
    "stage": compliance_record.stage,
    "word_count": compliance_record.compliance_output.word_count,
    "complete": compliance_record.compliance_output.completeness_check,
})
assert compliance_record.stage == "awaiting_compliance_review"
assert compliance_record.compliance_output.word_count <= 120

{'stage': 'awaiting_compliance_review', 'word_count': 57, 'complete': True}


## Human gate 2 — Approve SAR output

The final reviewer verifies narrative facts, completeness, and citations. Approval writes a structured JSON document; rejection would produce no filing.

In [4]:
compliance_decision = HumanDecisionInput(
    decision="Approved",
    reviewer="Notebook Compliance Reviewer",
    rationale="The narrative is factual, complete, within the word limit, and source-grounded",
)
filed_record = workflow.review_compliance(
    compliance_record.case.case_id,
    compliance_decision,
)
filed_path = Path(filed_record.filed_sar_path)
filed_document = json.loads(filed_path.read_text(encoding="utf-8"))
print({
    "stage": filed_record.stage,
    "file": filed_path.name,
    "filing_id": filed_document["filing_id"],
    "status": filed_document["status"],
})
assert filed_record.stage == "filed"
assert filed_document["risk_human_decision"]["decision"] == "Approved"
assert filed_document["compliance_human_decision"]["decision"] == "Approved"

{'stage': 'filed', 'file': 'CASE-6038471d-d01c-4595-96e0-7aa4ce85c0c7.json', 'filing_id': 'SAR-2b1a3c73-12fa-469c-9a5c-0073f6728036', 'status': 'approved'}


## Rejected path — Demonstrate staged cost control

A second case is rejected at the first human gate. The compliance agent is not called and no SAR document is created for that case.

In [5]:
rejected_risk = workflow.analyze_customer("CUST_0001")
rejected_record = workflow.review_risk(
    rejected_risk.case.case_id,
    HumanDecisionInput(
        decision="Rejected",
        reviewer="Notebook Reviewer",
        rationale="Observed activity does not support compliance escalation",
    ),
)
print({
    "stage": rejected_record.stage,
    "compliance_output": rejected_record.compliance_output,
    "filed_sar_path": rejected_record.filed_sar_path,
})
assert rejected_record.stage == "risk_rejected"
assert rejected_record.compliance_output is None
assert rejected_record.filed_sar_path is None

{'stage': 'risk_rejected', 'compliance_output': None, 'filed_sar_path': None}

## Auditability, timing, and staged cost metrics

Costs are explicit estimates. The key invariant is measured directly: compliance calls equal risk-approved cases, not all analyzed cases.

In [6]:
metrics = workflow.metrics()
metrics.model_dump(mode="json")

{'cases_analyzed': 2,
 'risk_approved': 1,
 'risk_rejected': 1,
 'compliance_generated': 1,
 'compliance_approved': 1,
 'compliance_rejected': 0,
 'filed_sars': 1,
 'avoided_compliance_calls': 1,
 'average_risk_processing_ms': 8.731299996725284,
 'average_compliance_processing_ms': 7.501099993532989,
 'staged_estimated_cost_usd': 0.008,
 'unstaged_estimated_cost_usd': 0.012,
 'estimated_cost_saved_usd': 0.004}

In [7]:
audit_frame = pd.DataFrame(workflow.audit_entries())
display(audit_frame[["timestamp", "component", "action", "case_id", "success", "user_decision"]])
print({
    "audit_entries": len(audit_frame),
    "filed_sars": metrics.filed_sars,
    "avoided_compliance_calls": metrics.avoided_compliance_calls,
    "estimated_cost_saved_usd": metrics.estimated_cost_saved_usd,
})
assert metrics.cases_analyzed == 2
assert metrics.risk_approved == 1
assert metrics.risk_rejected == 1
assert metrics.compliance_generated == 1
assert metrics.avoided_compliance_calls == 1
assert metrics.estimated_cost_saved_usd > 0

,timestamp,component,action,case_id,success,user_decision
0,2026-08-24T14:04:28.216834Z,RiskAnalyst,analyze_case,CASE-6038471d-d01c-4595-96e0-7aa4ce85c0c7,True,NaN
1,2026-08-24T14:04:28.224549Z,SarWorkflow,risk_review_requested,CASE-6038471d-d01c-4595-96e0-7aa4ce85c0c7,True,NaN
2,2026-08-24T14:04:28.256042Z,HumanReview,risk_decision,CASE-6038471d-d01c-4595-96e0-7aa4ce85c0c7,True,Approved
3,2026-08-24T14:04:28.259856Z,ComplianceOfficer,generate_narrative,CASE-6038471d-d01c-4595-96e0-7aa4ce85c0c7,True,NaN
4,2026-08-24T14:04:28.271119Z,SarWorkflow,compliance_review_requested,CASE-6038471d-d01c-4595-96e0-7aa4ce85c0c7,True,NaN
5,2026-08-24T14:04:28.304260Z,HumanReview,compliance_decision,CASE-6038471d-d01c-4595-96e0-7aa4ce85c0c7,True,Approved
6,2026-08-24T14:04:28.318146Z,SarWorkflow,sar_document_generated,CASE-6038471d-d01c-4595-96e0-7aa4ce85c0c7,True,Approved
7,2026-08-24T14:04:28.349643Z,RiskAnalyst,analyze_case,CASE-b15700a5-6120-47a2-9ba6-1031eb59ee56,True,NaN
8,2026-08-24T14:04:28.363449Z,SarWorkflow,risk_review_requested,CASE-b15700a5-6120-47a2-9ba6-1031eb59ee56,True,NaN
9,2026-08-24T14:04:28.387787Z,HumanReview,risk_decision,CASE-b15700a5-6120-47a2-9ba6-1031eb59ee56,True,Rejected


{'audit_entries': 10, 'filed_sars': 1, 'avoided_compliance_calls': 1, 'estimated_cost_saved_usd': 0.004}


## Integration result

- Both agents exchange validated Pydantic objects.
- Invalid state transitions fail rather than bypassing human authority.
- Compliance generation runs only for risk-approved cases.
- SAR JSON includes case identity, UTC metadata, subject/accounts, transaction summary, both agent outputs, both named human decisions, and audit-log location.
- JSONL audit, audit summary, workflow metrics, and filed SAR outputs are organized under `starter/outputs/`.
- The FastAPI/vanilla JavaScript application exposes the same workflow through interactive review controls.
